# Kairos · Этап 0 — CIDA-Loop против простого цикла

**Вопрос:** даёт ли латентное обсуждение агентов (CIDA) что-то сверх простого рекуррентного цикла (TRM)?

Задача — Sudoku-Extreme (как в HRM/TRM): 1000 обучающих головоломок с аугментациями, оценка на 5000 отложенных.

| Прогон | Что проверяет |
|---|---|
| `plain` | базовая линия: тот же цикл без агентов, почти те же параметры |
| `plain_compute` | тот же цикл, но в 4 раза больше латентных шагов — **те же FLOPs**, что у CIDA |
| `cida` | полная CIDA-Loop: 4 агента, связь, смещение к разногласию |
| `cida_nocomm` | агенты без связи (ансамбль внутри цикла) |
| `cida_nobias` | связь без смещения к разногласию |

**Правило решения (фиксируем ДО запуска):** CIDA считается победой, если на ≥3 зёрнах
`cida` лучше **и** `plain`, **и** `plain_compute` по точности больше чем на 2 std —
или при равной точности даёт заметно лучшую калибровку уверенности (AUROC/ECE).
Иначе честно записываем отрицательный результат и меняем идею.

In [ ]:
!nvidia-smi
import os
if not os.path.exists("CIDA-plugin"):
    !git clone https://github.com/Kairatzh/CIDA-plugin.git
%cd CIDA-plugin
!pip -q install huggingface_hub

## Настройки
Сначала запустите `PRESET = "quick"` — он покажет скорость (puzzles/s) на вашей GPU. По ней оцените время `full`.

In [ ]:
PRESET = "quick"   # "quick" | "full"
SEEDS = [0] if PRESET == "quick" else [0, 1, 2]

common = {
    "quick": "--iters 3000 --n-sup 8 --eval-every 1000 --n-test 2000",
    "full":  "--iters 20000 --n-sup 16 --eval-every 2000 --n-test 5000",
}[PRESET] + " --data extreme --n-train 1000 --compile"

EXPERIMENTS = {
    "plain":         "--variant plain",
    "plain_compute": "--variant plain --n-latent 24",
    "cida":          "--variant cida",
    "cida_nocomm":   "--variant cida --no-comm",
    "cida_nobias":   "--variant cida --no-bias",
}

In [ ]:
import os
from IPython import get_ipython
for seed in SEEDS:
    for name, flags in EXPERIMENTS.items():
        run = f"{name}_s{seed}"
        if os.path.exists(f"runs/{run}/result.json"):
            print("skip", run); continue
        # python -u + !-вызов: лог обучения виден в ячейке сразу, без буферизации
        cmd = f"python -u -m kairos_loop.train {flags} {common} --seed {seed} --name {run} --out runs/{run}"
        print(">>>", cmd, flush=True)
        get_ipython().system(cmd)
        if not os.path.exists(f"runs/{run}/result.json"):
            raise RuntimeError(f"{run} не завершился — смотрите лог выше")


## Результаты

In [ ]:
!python -m kairos_loop.report runs/*/result.json

In [ ]:
import glob, json
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
for path in sorted(glob.glob("runs/*_s0/result.json")):
    r = json.load(open(path))
    acc = r["final"]["exact_acc_by_step"]
    plt.plot(range(1, len(acc) + 1), acc, marker="o", label=r["name"])
plt.xlabel("шаг супервизии (сколько модель «думала»)")
plt.ylabel("точность (головоломка целиком)")
plt.legend(); plt.grid(alpha=.3); plt.title("Точность в зависимости от глубины рассуждения")
plt.show()

## Как читать
- **Exact** — доля полностью решённых отложенных головоломок (главная метрика).
- **Exact (adaptive) / Steps** — точность и средняя глубина, если модель сама решает, когда остановиться.
- **Halt AUROC / ECE** — насколько уверенность модели отличает верные ответы от неверных.
- **Agree AUROC** — то же для согласия агентов (только CIDA): это и есть «Kairos знает, когда не знает».

Сохраните `runs/` (Google Drive) — эти JSON и есть доказательная база для статьи и питча.